In [0]:
dbutils.widgets.text("business_date", "", "Business Date (YYYY-MM-DD, empty=latest)")

import sys
sys.path.append("/Workspace/Shared")
import aws_utils

env = aws_utils.get_env_prefix()

business_date_param = dbutils.widgets.get("business_date").strip()

if business_date_param:
    run_date = business_date_param
else:
    run_date = str(spark.sql(f"""
        SELECT MAX(BusinessDate) FROM `{env}-cdh`.primaryactive.application WHERE SourceSystem = 'fldt'
    """).collect()[0][0])

print(f"env:       {env}")
print(f"run_date:  {run_date}")

# FLDT Application Info

Replicates on-prem view `[PRC].[operation].[FLDT_applicationInfo]`.

## Table Mapping

| On-Prem | Cloud | Schema |
| --- | --- | --- |
| `DWH.act.Application` | `{env}-cdh.primaryactive.application` | Source |
| `DWH.inapp.Customer` | `{env}-cdh.derivativeinternal.inapp_customer` | Materialized |
| `DWH.inapp.Loan` | `{env}-cdh.derivativeinternal.inapp_loan` | Materialized |
| `DWH.inapp.UnifiedLoan` | `{env}-cdh.derivativeinternal.inapp_unifiedloan` | Materialized |
| `DWH.inapp.AccountImplInfo` | `{env}-cdh.derivativeinternal.inapp_accountimplinfo` | Materialized |
| `DWH.act.ApplicationDetails` | `{env}-cdh.derivativeinternal.act_applicationdetails` | Perimeter |
| `DWH.ref.Portfolio` | `{env}-cdh.primary.portfolio` | Lookup |

## Output

`{env}-cdh-delivery.it.fldt_applicationinfo` — daily overwrite

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{env}-cdh`.derivativeinternal.inapp_customer AS
    SELECT DISTINCT
        cust.Id,
        pty.PartyKey,
        prf.PortfolioId,
        cust.ApplicationId                                            AS SourceApplicationId,
        app.ApplicationID                                             AS ApplicationId,
        cust.CustomerCode,
        cust.PositionDate,
        cust.BusinessDate
    FROM `{env}-cdh`.landed.fldt_customer cust
    LEFT JOIN `{env}-cdh`.landed.fldt_application appl
      ON appl.Id = cust.ApplicationId
      AND appl.BusinessDate = cust.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.application app
      ON app.ApplicationKey = appl.ApplicationKey
      AND app.BusinessDate = cust.BusinessDate
    LEFT JOIN `{env}-cdh`.primary.portfolio prf
      ON prf.Prefix = LEFT(cust.CustomerCode, 3)
      AND prf.BusinessDate = cust.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.party pty
      ON pty.CDI = CASE 
          WHEN cust.CustomerCode LIKE '%CMS%' THEN SUBSTRING(cust.CustomerCode, 4, 50)
          ELSE cust.CustomerCode 
        END
      AND pty.BusinessDate = cust.BusinessDate
    WHERE cust.BusinessDate = {run_date!r}
""")
print(f"inapp_customer created ({run_date})")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{env}-cdh`.derivativeinternal.inapp_loan AS
    SELECT DISTINCT
        loan.Id,
        loan.AccountNumberWithPrefix                                  AS ExposureNaturalKey,
        exp.ExposureId,
        COALESCE(loan.ContractNumberSystemicNr, loan.ContractNumber)  AS AgreementNaturalKey,
        agr.AgreementId,
        prf.PortfolioId,
        pty.PartyKey,
        loan.CustomerId,
        loan.IsForModification,
        loan.NumberOfPreviousModifications,
        loan.AccountNumber,
        loan.AccountNumberWithPrefix,
        loan.ContractNumberSystemicNr,
        loan.BusinessDate
    FROM `{env}-cdh`.landed.fldt_loan loan
    INNER JOIN `{env}-cdh`.landed.fldt_customer cust
      ON cust.Id = loan.CustomerId
      AND cust.BusinessDate = loan.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.exposure exp
      ON exp.ExposureKey = SUBSTRING(loan.AccountNumberWithPrefix, 4)
      AND exp.BusinessDate = loan.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.agreement agr
      ON agr.AgreementKey = loan.ContractNumberSystemicNr
      AND agr.BusinessDate = loan.BusinessDate
    LEFT JOIN `{env}-cdh`.primary.portfolio prf
      ON prf.Prefix = LEFT(loan.AccountNumberWithPrefix, 3)
      AND prf.BusinessDate = loan.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.party pty
      ON pty.CDI = CASE 
          WHEN cust.CustomerCode LIKE '%CMS%' THEN SUBSTRING(cust.CustomerCode, 4, 50)
          ELSE cust.CustomerCode 
        END
      AND pty.BusinessDate = loan.BusinessDate
    WHERE loan.BusinessDate = {run_date!r}
""")
print(f"inapp_loan created ({run_date})")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{env}-cdh`.derivativeinternal.inapp_unifiedloan AS
    SELECT DISTINCT
        ul.Id,
        ul.LoanId,
        exp.ExposureId,
        agr.AgreementId,
        prf.PortfolioId,
        ul.AccountNumberWithPrefix,
        ul.ContractNumberSystemicNr,
        ul.BusinessDate
    FROM `{env}-cdh`.landed.fldt_unifiedloan ul
    LEFT JOIN `{env}-cdh`.primaryactive.exposure exp
      ON exp.ExposureKey = SUBSTRING(ul.AccountNumberWithPrefix, 4)
      AND exp.BusinessDate = ul.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.agreement agr
      ON agr.AgreementKey = ul.ContractNumberSystemicNr
      AND agr.BusinessDate = ul.BusinessDate
    LEFT JOIN `{env}-cdh`.primary.portfolio prf
      ON prf.Prefix = LEFT(ul.AccountNumberWithPrefix, 3)
      AND prf.BusinessDate = ul.BusinessDate
    WHERE ul.BusinessDate = {run_date!r}
""")
print(f"inapp_unifiedloan created ({run_date})")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{env}-cdh`.derivativeinternal.inapp_accountimplinfo AS
    SELECT DISTINCT
        aiinfo.Id,
        rst.RestructuringId,
        aiinfo.AccountImplementationInfoId,
        aiinfo.FinalModificationProductId,
        aiinfo.NewAccountNumber,
        newl.ExposureId                                               AS NewExposureId,
        aiinfo.CustomerCode,
        pty.PartyKey,
        aiinfo.ImplementationStatusId,
        aiinfo.ImplementationStatusDescription,
        aiinfo.IsActive,
        aiinfo.AccountNumber,
        oldl.ExposureId,
        aiinfo.ProductKey,
        aiinfo.DateCreated,
        ln.Id                                                         AS LoanId,
        aiinfo.BusinessDate
    FROM `{env}-cdh`.landed.fldt_accountimplementationinfo aiinfo
    INNER JOIN `{env}-cdh`.primary.portfolio p
      ON LEFT(aiinfo.CustomerCode, 3) = p.Prefix
      AND p.IsSingleTicketFlag = 0
      AND p.BusinessDate = aiinfo.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.exposure newl
      ON newl.ExposureKey = aiinfo.NewAccountNumber
      AND newl.BusinessDate = aiinfo.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.exposure oldl
      ON oldl.ExposureKey = aiinfo.AccountNumber
      AND oldl.BusinessDate = aiinfo.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.party pty
      ON pty.CDI = CASE 
          WHEN aiinfo.CustomerCode LIKE '%CMS%' THEN SUBSTRING(aiinfo.CustomerCode, 4, 50)
          ELSE aiinfo.CustomerCode 
        END
      AND pty.BusinessDate = aiinfo.BusinessDate
    LEFT JOIN `{env}-cdh`.landed.fldt_finalmodificationproduct sug
      ON aiinfo.ProductKey = sug.ProductKey
      AND sug.BusinessDate = aiinfo.BusinessDate
    INNER JOIN `{env}-cdh`.landed.fldt_loan ln
      ON sug.LoanId = ln.Id
      AND ln.BusinessDate = aiinfo.BusinessDate
    LEFT JOIN `{env}-cdh`.primaryactive.restructuring rst
      ON rst.OldExposureKey = CONCAT(LEFT(aiinfo.CustomerCode, 3), aiinfo.AccountNumber)
      AND rst.NewExposureKey = CONCAT(LEFT(aiinfo.CustomerCode, 3), aiinfo.NewAccountNumber)
      AND rst.BusinessDate = aiinfo.BusinessDate
    WHERE aiinfo.BusinessDate = {run_date!r}
""")
print(f"inapp_accountimplinfo created ({run_date})")

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE `{env}-cdh-delivery`.it.fldt_applicationinfo AS
    SELECT
        PortfolioId, ApplicationId, ApplicationKey, LoanId,
        AccountNumber, AccountNumberRfx, ApplicationImplementationStatus,
        ApplicationActivationDate, NumberOfPreviousModifications,
        ApplicationCancelledDate, ImplementationStatusDescription, BusinessDate
    FROM (
        SELECT
            pf.PortfolioId                                                             AS PortfolioId,
            app.ApplicationID                                                          AS ApplicationId,
            app.ApplicationKey                                                         AS ApplicationKey,
            COALESCE(loan.ExposureId, ul.ExposureId)                                   AS LoanId,
            SUBSTRING(
                COALESCE(loan.AccountNumberWithPrefix, ul.AccountNumberWithPrefix), 4
            )                                                                          AS AccountNumber,
            COALESCE(loan.AccountNumberWithPrefix, ul.AccountNumberWithPrefix)          AS AccountNumberRfx,
            app.ApplicationImplementationStatus                                        AS ApplicationImplementationStatus,
            app.ApplicationActivationDate                                              AS ApplicationActivationDate,
            loan.NumberOfPreviousModifications                                         AS NumberOfPreviousModifications,
            app.CancelledDate                                                          AS ApplicationCancelledDate,
            aii.ImplementationStatusDescription,
            app.BusinessDate                                                           AS BusinessDate,
            ROW_NUMBER() OVER (
                PARTITION BY app.ApplicationKey,
                    COALESCE(loan.AccountNumberWithPrefix, ul.AccountNumberWithPrefix)
                ORDER BY COALESCE(loan.ExposureId, ul.ExposureId)
            ) AS rn_dedup

        FROM `{env}-cdh`.primaryactive.application app

        LEFT JOIN `{env}-cdh`.primary.portfolio pf
            ON pf.Prefix = app.PortfolioKey
            AND pf.BusinessDate = app.BusinessDate

        INNER JOIN `{env}-cdh`.derivativeinternal.act_applicationdetails ad
            ON ad.ApplicationKey = app.ApplicationKey

        INNER JOIN (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY SourceApplicationId, CustomerCode 
                ORDER BY PositionDate DESC
            ) AS rn
            FROM `{env}-cdh`.derivativeinternal.inapp_customer
        ) customer 
            ON customer.ApplicationId = app.ApplicationID
            AND customer.CustomerCode = app.CDI
            AND customer.rn = 1

        INNER JOIN `{env}-cdh`.derivativeinternal.inapp_loan loan 
            ON loan.CustomerId = customer.Id 
            AND loan.IsForModification = true

        LEFT JOIN `{env}-cdh`.derivativeinternal.inapp_unifiedloan ul 
            ON ul.LoanId = loan.Id

        LEFT JOIN (
            SELECT *, ROW_NUMBER() OVER (
                PARTITION BY LoanId 
                ORDER BY DateCreated DESC
            ) AS rn2
            FROM `{env}-cdh`.derivativeinternal.inapp_accountimplinfo
            WHERE IsActive = true
        ) aii 
            ON aii.LoanId = loan.Id 
            AND aii.rn2 = 1

        WHERE app.SourceSystem = 'fldt'
          AND app.BusinessDate = {run_date!r}
    ) 
    WHERE rn_dedup = 1
""")

count = spark.sql(f"SELECT COUNT(*) AS cnt FROM `{env}-cdh-delivery`.it.fldt_applicationinfo").collect()[0]["cnt"]
print(f"fldt_applicationinfo written: {count:,} rows (env={env}, date={run_date})")

## Validation (2026-05-18)

| Metric | Value |
| --- | --- |
| On-Prem rows | 238,105 |
| Cloud rows | 238,262 |
| On-prem match rate | **100.00%** |
| Cloud extra | 157 (perimeter timing) |

All columns match >= 99.9% (null-safe).